In [ ]:
# Uncomment the following line to install required packages if needed:
# !pip install pandas numpy tqdm -q

In [ ]:
# Utility
from __future__ import annotations
from abc import ABC, abstractmethod
from tqdm import tqdm

# Data Manipulation
import pandas as pd
import numpy as np

In [ ]:
class CONFIG:
    VERSION = "0.5.0"
    RANDOM_STATE = 0
    SCALE = 1  # if SCALE=1, 600,000 records
    REFERENCE_DATE = pd.to_datetime("2026-3-31")

CONFIG.VERSION

## Inforce Policies

In [ ]:
class InforceGenerator:
    def __init__(
        self,
        product_name: str | None = None,
        product_code: str = "XX",
        female_ratio: float = 0.5,
        corporate_ratio: float = 0.5,
        conversion_ratio: float = 0.25,
        duration_year_min: int = 0,
        duration_year_max: int = 20,
        issue_age_min: int = 15,
        issue_age_max: int = 65,
        issue_age_alpha: int = 2,
        issue_age_beta: int = 5,
        attained_age_max: int = 70,
        nrs_score_max: float = 200,
        lapse_rate: float = 0.03,
        renewal_term: int | None = None,
        renewal_prob: float = 0.90,
        moral_risk_prob: float = 0.001,
        riders: list[Rider] | None = None,
        random_seed: int | None = None,
        simplified_issue: bool = False
    ):
        self.product_name = product_name
        self.product_code = product_code
        self.female_ratio = female_ratio
        self.corporate_ratio = corporate_ratio
        self.conversion_ratio = conversion_ratio
        self.duration_year_min = duration_year_min
        self.duration_year_max = duration_year_max
        self.issue_age_min = issue_age_min
        self.issue_age_max = issue_age_max
        self.issue_age_alpha = issue_age_alpha
        self.issue_age_beta = issue_age_beta
        self.attained_age_max = attained_age_max
        self.nrs_score_max = nrs_score_max
        self.lapse_rate = lapse_rate
        self.renewal_term = renewal_term
        self.renewal_prob = renewal_prob
        self.moral_risk_prob = moral_risk_prob
        self.riders = riders if riders else []
        self.rng = np.random.default_rng(seed=random_seed)
        self.simplified_issue = simplified_issue

    def generate(
        self,
        n_samples: int = 1000,
        reference_date: str | pd.Timestamp = "2026-3-31"
    ) -> pd.DataFrame:
        df_list = []
        current_count = 0
        if not isinstance(reference_date, pd.Timestamp):
            reference_date = pd.to_datetime(reference_date)

        if not isinstance(n_samples, int):
            n_samples = int(n_samples)

        while current_count < n_samples:
            batch_size = int((n_samples - current_count) * 2) + 100

            ## Gender
            gender = self.rng.choice(
                a=['Male', 'Female'],
                size=batch_size,
                p=[1 - self.female_ratio, self.female_ratio]
            )

            ## Corporate Indicator
            corporate_indicator = self.rng.binomial(
                n=1,
                p=self.corporate_ratio,
                size=batch_size
            )

            ## Conversion_indicator
            conversion_indicator = self.rng.choice(
                a=['New Business', 'Conversion'],
                p=[1 - self.conversion_ratio, self.conversion_ratio],
                size=batch_size
            )

            ## Sales Prefecture
            pref_weights = np.array([
              0.041, 0.010, 0.010, 0.018, 0.008, 0.009, 0.015, 0.023, 0.023, 0.016,
              0.057, 0.050, 0.111, 0.074, 0.018, 0.008, 0.009, 0.006, 0.007, 0.017,
              0.016, 0.029, 0.060, 0.014, 0.011, 0.020, 0.070, 0.045, 0.055, 0.011,
              0.005, 0.005, 0.015, 0.015, 0.011, 0.006, 0.007, 0.008, 0.011, 0.040,
              0.006, 0.010, 0.014, 0.009, 0.008, 0.013, 0.012
            ])
            pref_weights = pref_weights / pref_weights.sum()
            sales_prefecture = self.rng.choice(
                a=np.arange(1, 48),
                size=batch_size,
                p=pref_weights
            )

            ## Distribution Channel
            channel_weights = np.array([35, 15, 40, 10])
            channel_weights = channel_weights / channel_weights.sum()
            distribution_channel = self.rng.choice(
                a=np.arange(1, 5),
                size=batch_size,
                p=channel_weights
            )

            ## Issue Age
            issue_age = (
                self.issue_age_min
                + self.rng.beta(
                    a=self.issue_age_alpha,
                    b=self.issue_age_beta,
                    size=batch_size
                )
                * (self.issue_age_max - self.issue_age_min)
            )
            issue_age = issue_age.astype(dtype=int)

            ## Years Elapsed
            duration_year = self.rng.integers(
                low=self.duration_year_min,
                high=self.duration_year_max,
                size=batch_size
            )

            ## Attained Age
            attained_age = issue_age + duration_year

            ## Issue Date and Exposure
            days_elapsed = self.rng.integers(
                low=1,
                high=366,
                size=batch_size
            )
            issue_date = np.array([
                reference_date - pd.DateOffset(days=d, years=int(y))
                for d, y in zip(days_elapsed, duration_year)
            ])
            exposure = np.where(
                duration_year == 0, days_elapsed / 365.0, 1.0
            )

            ## Health and NRS Score
            init_z = self.rng.normal(size=batch_size)
            rho = .90
            current_z = (
                (rho ** duration_year) * init_z
                + np.sqrt(1.0 - rho ** (2 * duration_year))
                * self.rng.normal(size=batch_size)
            )

            def health_vol(age):
                return 40.0 + age / 2.0 + (np.clip(age - 30, 0, 100) ** 2) / 50.0

            init_vol = health_vol(issue_age)
            current_vol = health_vol(attained_age)

            def map_to_lognormal(z, vol, expected_mean=100.0, shift=50.0):
                adjusted_mean = expected_mean - shift
                cv = vol / adjusted_mean
                omega = np.sqrt(np.log(1.0 + cv ** 2))
                mu = np.log(adjusted_mean) - 0.5 * (omega ** 2)
                return np.exp(mu + omega * z) + shift

            initial_health_score = map_to_lognormal(init_z, init_vol)
            current_health_score = map_to_lognormal(current_z, current_vol)
            current_health_score = np.clip(current_health_score, 50, 500)

            ## Medical Underwriting
            w_matrix = np.zeros((batch_size, 3), dtype=float)
            if self.simplified_issue:
                w_matrix[:, 0] = 1.0 # Declaration
                w_matrix[:, 1] = 0.0
                w_matrix[:, 2] = 0.0
            else:
                w_matrix[:, 0] = np.maximum(30, 90 - issue_age) # Declaration
                w_matrix[:, 1] = 45 # Health Check-Up
                w_matrix[:, 2] = np.maximum(3, issue_age / 3 - 10) # Medical Investigation
            p_matrix = w_matrix / w_matrix.sum(axis=1, keepdims=True)
            underwriting_method = np.argmax(
                self.rng.random((batch_size, 1)) < np.cumsum(p_matrix, axis=1), axis=1
                ) + 1
            uw_error_std = np.where(
                underwriting_method == 1,
                20.0, np.where(underwriting_method == 2, 3.0, 5.0)
            )
            uw_error_mean = np.where(underwriting_method == 1, 0.0, 0.0)
            underwriting_error = self.rng.normal(
                loc=uw_error_mean, scale=uw_error_std, size=batch_size
            )
            observed_health = initial_health_score + underwriting_error
            nrs_score = np.round(np.maximum(observed_health, 100), -1).astype(int)

            ## Moral Risk Flag
            moral_risk_flag = self.rng.binomial(
                n=1,
                p=self.moral_risk_prob,
                size=batch_size
            )

            ## Policy ID
            sequence = np.arange(current_count + 1, current_count + batch_size + 1)
            _seq_str = np.char.zfill(sequence.astype(str), 6)
            policy_id = self.product_code + "-" + _seq_str

            ## Dropout Before BoY
            def gompertz_makeham(x, A, B, C, l0=100000):
                lx = l0 * np.exp(-A * x - (B / C) * (np.exp(C * x) - 1))
                return lx

            A_arr = 0.00036827304
            B_arr = np.where(gender == 'Male', 6.499415264e-06, 3.249707632e-06)
            C_arr = 0.11069899616

            is_inforce = duration_year > 0

            lapse_rate_BoY = np.clip(self.lapse_rate * np.sqrt(200 / (initial_health_score + current_health_score)), 0.0, 1.0)
            not_lapsed_BoY = np.where(is_inforce, (1 - lapse_rate_BoY) ** (duration_year + days_elapsed / 365.0 - 1), 1.0)

            if self.renewal_term is None:
                renewed_BoY = 1.0
            else:
                renewed_BoY = np.where(is_inforce, self.renewal_prob ** ((duration_year - 1) // self.renewal_term), 1.0)

            lx_issue = gompertz_makeham(issue_age, A_arr, B_arr, C_arr)
            lx_BoY = gompertz_makeham(attained_age - 1, A_arr, B_arr, C_arr)
            survived_BoY = np.where(is_inforce, np.clip(lx_BoY / lx_issue, 0.0, 1.0), 1.0)

            dropout_prob_BoY = np.clip(1.0 - (not_lapsed_BoY * renewed_BoY * survived_BoY), 0.0, 1.0)

            ## Dropout Before EoY
            lapse_rate_EoY = np.clip(self.lapse_rate * np.sqrt(100 / current_health_score), 0.0, 1.0)
            not_lapsed_EoY = (1 - lapse_rate_EoY)

            if self.renewal_term is None:
                renewed_EoY = 1.0
            else:
                is_renewal_year = is_inforce & ((duration_year % self.renewal_term) == 0)
                renewed_EoY = np.where(is_renewal_year, self.renewal_prob, 1.0)

            lx_EoY = gompertz_makeham(attained_age, A_arr, B_arr, C_arr)
            survived_EoY = np.clip(lx_EoY / lx_BoY, 0.0, 1.0)

            dropout_prob_EoY = np.clip(1.0 - (not_lapsed_EoY * renewed_EoY * survived_EoY), 0.0, 1.0)

            dropout_EoY = self.rng.random(size=batch_size) <= dropout_prob_EoY

            ## Exposure
            exposure_fraction = self.rng.random(size=batch_size)
            exposure = np.where(dropout_EoY, exposure * exposure_fraction, exposure)

            ## Renewal Status
            renewal_thresh = 100 if self.renewal_term is None else self.renewal_term
            renewal_status = np.where(
                duration_year < renewal_thresh,
                'Pre-Renewal', 'Post-Renewal'
            )

            valid_mask = (
                (issue_age >= self.issue_age_min) &
                (issue_age <= self.issue_age_max) &
                (attained_age <= self.attained_age_max) &
                (nrs_score <= self.nrs_score_max) &
                (self.rng.random(size=batch_size) > dropout_prob_BoY)
            )

            df_temp = pd.DataFrame(data={
                'policy_id': policy_id[valid_mask],
                'base_product': self.product_name,
                'reference_date': reference_date,
                'issue_date': issue_date[valid_mask],
                'gender': gender[valid_mask],
                'issue_age': issue_age[valid_mask],
                'duration_year': duration_year[valid_mask],
                'attained_age': attained_age[valid_mask],
                'underwriting_method': underwriting_method[valid_mask],
                'conversion_indicator': conversion_indicator[valid_mask],
                'corporate_indicator': corporate_indicator[valid_mask],
                'sales_prefecture': sales_prefecture[valid_mask],
                'distribution_channel': distribution_channel[valid_mask],
                'nrs_score': nrs_score[valid_mask],
                'initial_health_score': initial_health_score[valid_mask],
                'current_health_score': current_health_score[valid_mask],
                'moral_risk_flag': moral_risk_flag[valid_mask],
                'renewal_status': renewal_status[valid_mask],
                'exposure': exposure[valid_mask]
            })

            df_list.append(df_temp)
            current_count += len(df_temp)

        df_final = pd.concat(df_list, ignore_index=True).iloc[:n_samples].copy()

        for rider in self.riders:
            df_final = rider.attach(df_final)

        return df_final

## Riders

In [ ]:
class Rider(ABC):
    column_name: str = "rider"
    nrs_score_max: int = 120

    def __init__(self, random_seed: int | None = None):
        self.rng = np.random.default_rng(seed=random_seed)

    def attach(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        base_prob = self._calc_prob(df=df)
        if 'nrs_score' in df.columns:
            base_prob = np.where(df['nrs_score'] > self.nrs_score_max, 0.0, base_prob)
        attached = self.rng.binomial(n=1, p=base_prob)
        values = self._calc_amount(df=df)
        df[self.column_name] = np.where(attached == 1, values, 0)
        return df

    @abstractmethod
    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
        pass

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
        return np.ones(shape=len(df), dtype=int)

In [ ]:
class HospLumpSumRider(Rider):
    column_name = "sum_assured"
    nrs_score_max: int = 500

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
        return np.ones(len(df))

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
        attained_age = df['attained_age'].values
        health = df['initial_health_score'].values
        moral = df['moral_risk_flag'].values
        batch_size = len(df)
        true_risk = (health - 100) + (moral * 100)

        age_knots = np.array([20, 25, 30, 37, 45, 55, 65, 80])
        median_knots_man = np.array([10.0, 15.0, 25.0, 30.0, 25.0, 15.0, 10.0, 10.0])
        mu_age = np.interp(attained_age, age_knots, np.log(median_knots_man))

        risk_beta = 0.005
        mu = mu_age + (true_risk * risk_beta)

        sigma = 0.55
        log_amount = self.rng.normal(loc=mu, scale=sigma, size=batch_size)
        raw_amount_man = np.exp(log_amount)

        anchor_prob = 0.15
        is_anchored = self.rng.random(batch_size) < anchor_prob

        anchored_man = np.clip(np.round(raw_amount_man / 5.0) * 5.0, 5.0, 40.0)
        fine_man = np.clip(np.round(raw_amount_man), 1.0, 40.0)
        selected_amounts_man = np.where(is_anchored, anchored_man, fine_man).astype(int)

        return selected_amounts_man * 10000

In [ ]:
class MedicalCompBenefitRider(Rider):
    column_name = "sum_assured_medical_comp_benefit_rider"
    nrs_score_max: int = 500

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
        return np.where(df['base_product'].astype(str).str.contains('Medical'), 0.0, 1.0)

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
        return self.rng.choice([5000, 10000, 20000], size=len(df))

class CancerRider(Rider):
    column_name = "sum_assured_cancer_rider"
    nrs_score_max = 100

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
        base_prob = 0.6 - ((df['issue_age'] - 45) ** 2) * 0.0005
        return np.clip(base_prob * np.exp((df['initial_health_score'] - 100) / 250), 0.0, 0.8)

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
        return self.rng.choice([500000, 1000000], size=len(df))

class NursingCareRider(Rider):
    column_name = "sum_assured_nursing_care_rider"
    nrs_score_max = 110

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
        return np.clip(0.1 + (df['issue_age'] / 200), 0.0, 0.5)

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
        return self.rng.choice([500000, 1000000], size=len(df))

class DementiaRider(Rider):
    column_name = "sum_assured_dementia_rider"
    nrs_score_max = 110

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
       return np.full(len(df), 0.5)

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
       return self.rng.choice([500000, 1000000], size=len(df))

class FemaleDiseasesRider(Rider):
    column_name = "sum_assured_female_rider"

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
       return np.where(df['gender'] == 'Female', 0.75, 0.0)

    def _calc_amount(self, df: pd.DataFrame) -> np.ndarray:
       return self.rng.choice([5000, 10000], size=len(df))

class AccidentalInjuryRider(Rider):
    column_name = "sum_assured_injury_rider"
    nrs_score_max: int = 500

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
       return np.full(len(df), 0.80)

class PremWaiverRider(Rider):
    column_name = "premium_waiver_rider"
    nrs_score_max = 100

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
        return np.clip((0.7 - df['issue_age'] * 0.01) * np.exp((df['initial_health_score'] - 100) / 250), 0.0, 1.0)

class AdvancedMedicalRider(Rider):
    column_name = "advanced_medical_rider"
    nrs_score_max = 120

    def _calc_prob(self, df: pd.DataFrame) -> np.ndarray:
       return np.full(len(df), 0.8)

In [ ]:
# parameters for each product category

CATEGORY_KWARGS = {
    "Term": {
        "female_ratio": 0.45, "issue_age_min": 20, "issue_age_max": 60,
        "issue_age_alpha": 3.0, "issue_age_beta": 4.0, "attained_age_max": 80,
        "nrs_score_max": 105, "moral_risk_prob": 0.002
    },
    "Renewable Term": {
        "female_ratio": 0.40, "issue_age_min": 20, "issue_age_max": 60,
        "issue_age_alpha": 3.0, "issue_age_beta": 6.0, "attained_age_max": 70,
        "nrs_score_max": 105, "renewal_term": 5, "moral_risk_prob": 0.005
    },
    "Medical": {
        "female_ratio": 0.52, "issue_age_min": 20, "issue_age_max": 70,
        "issue_age_alpha": 3.0, "issue_age_beta": 5.0, "attained_age_max": 99,
        "nrs_score_max": 120, "moral_risk_prob": 0.005
    },
    "Whole Life": {
        "female_ratio": 0.50, "issue_age_min": 20, "issue_age_max": 65,
        "issue_age_alpha": 3.0, "issue_age_beta": 3.0, "attained_age_max": 99,
        "nrs_score_max": 120, "moral_risk_prob": 0.001
    },
    "Cancer": {
        "female_ratio": 0.55, "issue_age_min": 20, "issue_age_max": 60,
        "issue_age_alpha": 3.0, "issue_age_beta": 3.0, "attained_age_max": 80,
        "nrs_score_max": 105, "moral_risk_prob": 0.002
    }
}

# mappings for product categories

CATEGORY_MAPPINGS = {
    "Prefix": {"Te": "Term", "Re": "Renewable Term", "Wh": "Whole Life", "Me": "Medical", "Ca": "Cancer"},
    "CodeOffset": {"Term": 0, "Renewable Term": 1, "Whole Life": 2, "Medical": 3, "Cancer": 4},
    "Weight": {"Term": 0.6, "Renewable Term": 0.8, "Whole Life": 1.2, "Medical": 1.0, "Cancer": 0.6}
}

## Generate Policies

In [ ]:
product_names = [
    "Term1992", "Term2001", "Term2008", "Term2016", "Term2022",
    "RenewableTerm1998", "RenewableTerm2007", "RenewableTerm2015", "RenewableTerm2021",
    "WholeLife1994", "WholeLife2003", "WholeLife2012", "WholeLife2020",
    "Medical2000", "Medical2008", "Medical2014", "Medical2020_A", "Medical2020_B", "Medical2024_A", "Medical2024_B",
    "Cancer1997", "Cancer2009", "Cancer2021"
]

base_products = pd.DataFrame({"ProductName": product_names})
base_products["ProductCategory"] = base_products["ProductName"].str[:2].map(CATEGORY_MAPPINGS["Prefix"])
base_products["Year"] = base_products["ProductName"].str.extract(r'(\d{4})').astype(int)

base_products["Code"] = (
    base_products.groupby("ProductCategory").cumcount() + 1
    + base_products["ProductCategory"].map(CATEGORY_MAPPINGS["CodeOffset"]) * 10
).astype(str).str.zfill(2)

unique_years = (
    base_products[["ProductCategory", "Year"]]
    .drop_duplicates()
    .sort_values(["ProductCategory", "Year"])
)
unique_years["NextYear"] = unique_years.groupby("ProductCategory")["Year"].shift(-1)

base_products = base_products.merge(unique_years, on=["ProductCategory", "Year"], how="left")
base_products["duration_year_max"] = CONFIG.REFERENCE_DATE.year - base_products["Year"]
base_products["duration_year_min"] = np.where(
    base_products["NextYear"].isna(), 0, CONFIG.REFERENCE_DATE.year - base_products["NextYear"] + 1
).astype(int)
base_products = base_products.drop(columns=["NextYear"])

duration = base_products["duration_year_max"] - base_products["duration_year_min"] + 1
adjusted_duration = np.where(base_products["duration_year_min"] == 0, duration - 0.5, duration)
base_products["CategoryWeight"] = base_products["ProductCategory"].map(CATEGORY_MAPPINGS["Weight"])
base_products["GenerationWeight"] = base_products["Year"] / 1683.267

raw_policy_count = (
    3776
    * adjusted_duration
    * base_products["CategoryWeight"]
    * base_products["GenerationWeight"]
).astype(int)

base_products["policy_count"] = np.round(raw_policy_count * CONFIG.SCALE, 0).astype(int)
base_products

In [ ]:
df_list = []

for _, row in tqdm(base_products.iterrows(), total=len(base_products), desc="Generating Inforce"):
    p_name = str(row["ProductName"])
    p_code = str(row["Code"])
    p_type = str(row["ProductCategory"])

    kwargs = CATEGORY_KWARGS[p_type].copy()

    if p_name.endswith("_B"):
        kwargs.update({"nrs_score_max": 145, "moral_risk_prob": 0.010, "simplified_issue": True})
    elif p_name.endswith("_A"):
        kwargs.update({"nrs_score_max": 105, "moral_risk_prob": 0.004})

    kwargs.update({
        "product_name": p_name,
        "product_code": p_code,
        "duration_year_max": int(row["duration_year_max"]),
        "duration_year_min": int(row["duration_year_min"]),
        "random_seed": int(p_code)
    })

    base_seed = int(p_code) * 100 + CONFIG.RANDOM_STATE
    kwargs["riders"] = [
        MedicalCompBenefitRider(base_seed),
        HospLumpSumRider(base_seed + 1),
        CancerRider(base_seed + 2),
        NursingCareRider(base_seed + 3),
        DementiaRider(base_seed + 4),
        FemaleDiseasesRider(base_seed + 5),
        AccidentalInjuryRider(base_seed + 6),
        PremWaiverRider(base_seed + 7),
        AdvancedMedicalRider(base_seed + 8)
    ]

    df_temp = InforceGenerator(**kwargs).generate(n_samples=row["policy_count"])

    df_temp["base_product_category"] = p_type
    df_temp["base_product_code"] = p_code

    df_list.append(df_temp)

df_all = (
    pd.concat(df_list, ignore_index=True)
    .sample(frac=1, random_state=CONFIG.RANDOM_STATE)
    .reset_index(drop=True)
)

In [ ]:
PREFECTURE_NAMES = {
    1: 'Hokkaido', 2: 'Aomori', 3: 'Iwate', 4: 'Miyagi', 5: 'Akita', 6: 'Yamagata', 7: 'Fukushima',
    8: 'Ibaraki', 9: 'Tochigi', 10: 'Gunma', 11: 'Saitama', 12: 'Chiba', 13: 'Tokyo', 14: 'Kanagawa',
    15: 'Niigata', 16: 'Toyama', 17: 'Ishikawa', 18: 'Fukui', 19: 'Yamanashi', 20: 'Nagano',
    21: 'Gifu', 22: 'Shizuoka', 23: 'Aichi', 24: 'Mie', 25: 'Shiga', 26: 'Kyoto', 27: 'Osaka',
    28: 'Hyogo', 29: 'Nara', 30: 'Wakayama', 31: 'Tottori', 32: 'Shimane', 33: 'Okayama', 34: 'Hiroshima',
    35: 'Yamaguchi', 36: 'Tokushima', 37: 'Kagawa', 38: 'Ehime', 39: 'Kochi', 40: 'Fukuoka',
    41: 'Saga', 42: 'Nagasaki', 43: 'Kumamoto', 44: 'Oita', 45: 'Miyazaki', 46: 'Kagoshima', 47: 'Okinawa'
}

AREA_MAPPING = {
    'Hokkaido': 'Hokkaido',
    'Aomori': 'Tohoku', 'Iwate': 'Tohoku', 'Miyagi': 'Tohoku', 'Akita': 'Tohoku', 'Yamagata': 'Tohoku', 'Fukushima': 'Tohoku',
    'Ibaraki': 'Kanto', 'Tochigi': 'Kanto', 'Gunma': 'Kanto', 'Saitama': 'Kanto', 'Chiba': 'Kanto', 'Tokyo': 'Kanto', 'Kanagawa': 'Kanto',
    'Niigata': 'Chubu', 'Toyama': 'Chubu', 'Ishikawa': 'Chubu', 'Fukui': 'Chubu', 'Yamanashi': 'Chubu', 'Nagano': 'Chubu', 'Gifu': 'Chubu', 'Shizuoka': 'Chubu', 'Aichi': 'Chubu',
    'Mie': 'Kinki', 'Shiga': 'Kinki', 'Kyoto': 'Kinki', 'Osaka': 'Kinki', 'Hyogo': 'Kinki', 'Nara': 'Kinki', 'Wakayama': 'Kinki',
    'Tottori': 'Chugoku', 'Shimane': 'Chugoku', 'Okayama': 'Chugoku', 'Hiroshima': 'Chugoku', 'Yamaguchi': 'Chugoku',
    'Tokushima': 'Shikoku', 'Kagawa': 'Shikoku', 'Ehime': 'Shikoku', 'Kochi': 'Shikoku',
    'Fukuoka': 'Kyushu', 'Saga': 'Kyushu', 'Nagasaki': 'Kyushu', 'Kumamoto': 'Kyushu', 'Oita': 'Kyushu', 'Miyazaki': 'Kyushu', 'Kagoshima': 'Kyushu', 'Okinawa': 'Kyushu'
}

RIDER_COLS = [
    'sum_assured', 'sum_assured_medical_comp_benefit_rider', 'sum_assured_cancer_rider',
    'sum_assured_nursing_care_rider', 'sum_assured_dementia_rider', 'sum_assured_female_rider',
    'sum_assured_injury_rider', 'premium_waiver_rider', 'advanced_medical_rider'
]

df_all = df_all.copy()
rng = np.random.default_rng(seed=42)

df_all['number_of_riders'] = (df_all[RIDER_COLS] > 0).sum(axis=1)

df_all['sales_prefecture'] = df_all['sales_prefecture'].map(PREFECTURE_NAMES)
df_all['sales_prefecture_group'] = df_all['sales_prefecture'].map(AREA_MAPPING)

df_all['underwriting_method'] = df_all['underwriting_method'].map({1: 'Declaration', 2: 'Health Check-up', 3: 'Medical Examination'})
df_all['corporate_indicator'] = df_all['corporate_indicator'].map({0: 'No', 1: 'Yes'})
df_all['distribution_channel'] = df_all['distribution_channel'].map({1: 'CareerAgent', 2: 'BankAgency', 3: 'IndependentAgency', 4: 'DirectMarketing'})

for col in ['premium_waiver_rider', 'advanced_medical_rider']:
    df_all[col] = df_all[col].map({1: 'Yes', 0: 'No'})

has_rider = df_all['number_of_riders'] > 0
timing_choices = rng.choice(['At-Issue', 'After-Issue'], size=len(df_all), p=[0.99, 0.01])
df_all['rider_attachment_timing'] = np.where(has_rider, timing_choices, 'None')

age_bins = np.arange(15, 106, 5)
age_labels = [f"{age_bins[i]}-{age_bins[i]+4}" for i in range(len(age_bins)-1)]
df_all['attained_age_band'] = pd.cut(df_all['attained_age'], bins=age_bins, labels=age_labels, right=False)

elapsed_strings = np.where(df_all['duration_year'] >= 15, "15+", df_all['duration_year'].astype(str))
elapsed_order = [str(x) for x in range(15)] + ["15+"]
df_all['duration_year_band'] = pd.Categorical(elapsed_strings, categories=elapsed_order, ordered=True)

df_all['sum_assured_band'] = pd.cut(
    df_all['sum_assured'],
    bins=[0, 100000, 200000, 300000, 400000],
    labels=['1-10', '11-20', '21-30', '31-40']
)

categorical_cols = [
    'base_product', 'gender', 'underwriting_method', 'conversion_indicator',
    'corporate_indicator', 'sales_prefecture', 'sales_prefecture_group',
    'distribution_channel', 'renewal_status', 'rider_attachment_timing',
    'premium_waiver_rider', 'advanced_medical_rider', 'attained_age_band', 'sum_assured_band'
]

df_all[categorical_cols] = df_all[categorical_cols].astype('category')

## Simulate Payments

In [ ]:
def calculate_crude_rate(df: pd.DataFrame) -> np.ndarray:
    age = df['attained_age']
    age_bins = np.array([k*5+2 for k in range(19)])
    rates_male = np.array([
        0.0451, 0.0118, 0.0109, 0.0139, 0.0163, 0.0166, 0.0205, 0.0241, 0.0320, 0.0410,
        0.0595, 0.0849, 0.1195, 0.1598, 0.2138, 0.2791, 0.3798, 0.5521, 0.7753
    ])
    rates_female = np.array([
        0.0412, 0.0093, 0.0105, 0.0147, 0.0197, 0.0313, 0.0415, 0.0370, 0.0318, 0.0369,
        0.0480, 0.0646, 0.0847, 0.1122, 0.1525, 0.2175, 0.3384, 0.5191, 0.7473
    ])

    base_rate_male = np.interp(age, age_bins, rates_male)
    base_rate_female = np.interp(age, age_bins, rates_female)

    crude_rate = np.where(df['gender'] == 'Male', base_rate_male, base_rate_female)
    return crude_rate


def calculate_true_base_rate(df: pd.DataFrame) -> np.ndarray:
    age = df['attained_age'].values
    gender = df['gender'].values
    crude = df['crude_rate'].values

    age_slope = 0.85 + 0.15 * np.clip((age - 20) / 60.0, 0.0, 1.0)
    gender_slope = np.where(gender == 'Female', 1.10, 0.92)

    return crude * age_slope * gender_slope


def generate_payment_dataset(df: pd.DataFrame, random_seed: int = 0) -> pd.DataFrame:
    df = df.copy()
    rng = np.random.default_rng(seed=random_seed)

    df['crude_rate'] = calculate_crude_rate(df)
    safety_margin = 0.25
    df['pricing_rate'] = df['crude_rate'] * (1.0 + safety_margin)
    true_base_rate = calculate_true_base_rate(df)

    selection_factor = 1.0 - 0.45 * np.exp(-df['duration_year'].values / 2.5)

    health_factor = np.clip(1.0 + (df['current_health_score'].values - 100) / 100.0, 0.6, 2.2)
    moral_factor = np.where(df['moral_risk_flag'] == 1, 1.8, 1.0)

    amount_ratio = df['sum_assured'].values / 400000.0
    amount_main_effect = 1.0 + (amount_ratio ** 2.0) * 0.50
    early_duration = np.exp(-df['duration_year'].values / 3.0)
    adverse_selection_synergy = 1.0 + (early_duration * (amount_ratio ** 1.4)) * 0.45

    channel_effects = {'DirectMarketing': 1.08, 'IndependentAgency': 1.04, 'CareerAgent': 0.98, 'BankAgency': 0.95}
    channel_factor = df['distribution_channel'].astype(str).map(channel_effects).fillna(1.0).values

    # Product
    prod_effects = {'Medical': 1.08, 'Renewable Term': 1.03, 'Cancer': 1.00, 'Term': 0.95, 'Whole Life': 0.90}
    prod_factor = df['base_product_category'].astype(str).map(prod_effects).fillna(1.0).values

    rider_risk = 1.0 + 0.03 * (
        (df['sum_assured_nursing_care_rider'].values > 0).astype(int) +
        (df['sum_assured_dementia_rider'].values > 0).astype(int) +
        (df['premium_waiver_rider'].astype(str) == 'Yes').astype(int)
    )

    df['actual_rate'] = np.clip(
        true_base_rate * selection_factor * health_factor * moral_factor *
        amount_main_effect * adverse_selection_synergy * channel_factor * prod_factor * rider_risk,
        0.0, 3.0
    )

    # Simulate claims
    adjusted_lambda = df['actual_rate'] * df['exposure']
    df['actual_claim_count'] = rng.poisson(lam=adjusted_lambda)
    df['actual_claim_amount'] = df['actual_claim_count'] * df['sum_assured']

    df['expected_claim_count'] = df['pricing_rate'] * df['exposure']
    df['expected_claim_amount'] = df['expected_claim_count'] * df['sum_assured']

    return df

In [ ]:
df_all = generate_payment_dataset(df_all, random_seed=CONFIG.RANDOM_STATE)

df_all.shape

In [ ]:
# Dataset B Seriatim
cols_B = [
    'policy_id',
    'gender',
    'attained_age',
    'duration_year',
    'conversion_indicator',
    'renewal_status',
    'base_product',
    'corporate_indicator',
    'sum_assured',
    'sum_assured_medical_comp_benefit_rider',
    'sum_assured_cancer_rider',
    'sum_assured_nursing_care_rider',
    'sum_assured_dementia_rider',
    'sum_assured_female_rider',
    'sum_assured_injury_rider',
    'premium_waiver_rider',
    'advanced_medical_rider',
    'number_of_riders',
    'actual_claim_amount',
    'expected_claim_amount',
    'actual_claim_count',
    'expected_claim_count',
    'exposure',
    'underwriting_method',
    'distribution_channel',
    'sales_prefecture_group'
]

df_B = df_all[cols_B].copy()

In [ ]:
df_B.columns

In [ ]:
# Dataset A Aggregated
df_A_base = df_all[df_all['sum_assured'] > 0].copy()

df_A_base['observation_year'] = 2025
df_A_base['rider_product'] = '05'

# Exposure Sum Assured
df_A_base['exposure_amount'] = df_A_base['exposure'] * df_A_base['sum_assured']

# Grouping Keys
group_keys = [
    'observation_year',
    'rider_product',
    'gender',
    'attained_age_band',
    'duration_year_band',
    'conversion_indicator',
    'rider_attachment_timing',
    'renewal_status',
    'sum_assured_band',
    'base_product_category',
    'underwriting_method',
    'distribution_channel',
    'sales_prefecture_group',
    'corporate_indicator',
]

# !!! observed=True is important for saving memory
df_A = df_A_base.groupby(group_keys, observed=True).agg(
    exposure_amount=('exposure_amount', 'sum'),
    actual_claim_amount=('actual_claim_amount', 'sum'),
    expected_claim_amount=('expected_claim_amount', 'sum'),
    actual_claim_count=('actual_claim_count', 'sum'),
    expected_claim_count=('expected_claim_count', 'sum'),
    exposure=('exposure', 'sum')
).reset_index()

df_A = df_A[df_A['exposure_amount'] > 0].reset_index(drop=True)

In [ ]:
df_A.columns

In [ ]:
import os

# 1. Define file names (CSV format)
file_name_A = f'inforce_policies_aggregated_{CONFIG.VERSION}.csv'
file_name_B = f'inforce_policies_seriatim_{CONFIG.VERSION}.csv'
file_name_all = f'inforce_policies_full_{CONFIG.VERSION}.csv'

# --- Save to the same folder as the notebook ---
try:
    path_A = file_name_A
    path_B = file_name_B
    path_all = file_name_all

    # Export DataFrames to CSV format
    df_A.to_csv(path_A, index=False)
    df_B.to_csv(path_B, index=False)
    df_all.to_csv(path_all, index=False)

    print("=== Saved to Current Folder (CSV) ===")
    print(f"Dataset A: {path_A}")
    print(f"Dataset B: {path_B}")
    print(f"Dataset Full: {path_all}")

except Exception as e:
    print(f"An error occurred while saving files: {e}")